In [ ]:
!pip install dotenv openai pandas openpyxl requests chromadb

In [1]:
from agno.agent import Agent
from agno.models.message import Message
from agno.db.sqlite import SqliteDb
from agno.tools.serper import SerperTools
from agno.models.azure import AzureOpenAI
from agno.vectordb.chroma import ChromaDb
from agno.knowledge.chunking.row import RowChunking
from agno.knowledge.embedder.azure_openai import AzureOpenAIEmbedder
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.reader.excel_reader import ExcelReader

import os, json
import pandas as pd
from dotenv import load_dotenv  
from os import getenv
from pathlib import Path  

In [2]:
load_dotenv()
os.environ["AZURE_OPENAI_API_KEY"] = getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_DEPLOYMENT"] = getenv("LLM_MODEL")

In [3]:
model = AzureOpenAI(id=getenv("LLM-MODEL"), 
                    api_version=getenv("LLM_MODEL_VERSION"),
                    azure_endpoint=getenv("AZURE_INFERENCE_ENDPOINT"))
response = model.response(
    messages=[
        Message(role="user", content="Hello")
    ]
)
print(response.content)

Hello! How can I assist you today?


In [4]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   # <-- change to your folder

USE_CASE_FILE = DATA_FOLDER / 'use_case_Movie_Recommendation_v2_with_fi_Web_RAG_TAG.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'

# SQLite file for Agno session memory
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'
agent_db = SqliteDb(db_file=MEMORY_DB)


AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT=os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION=os.getenv("LLM_MODEL_VERSION") 
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

LLM_MODEL        = os.getenv("LLM_MODEL")
#EMBED_MODEL      = 'text-embedding-3-small'
MAX_ABT_RESULTS  = 5
MAX_TAG_RESULTS  = 10
RAG_TOP_K        = 4
SESSION_ID       = 'movie-bot-session-001'  # change per user

print(f'Data folder: {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')

Data folder: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/Agno-Class-exercises/data/input
  ✅  use_case_Movie_Recommendation_v2_with_fi_Web_RAG_TAG.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


In [14]:
df_full_abt = pd.read_excel(ABT_FILE)
df_abt_batch_1 = df_full_abt[0:1000]
df_abt_batch_1.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_1.xlsx'))
ABT_BATCH_1 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_1.xlsx')
df_abt_batch_2 = df_full_abt[1000:2000]
df_abt_batch_2.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_2.xlsx'))
ABT_BATCH_2 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_2.xlsx')
df_abt_batch_3 = df_full_abt[2000:3000]
df_abt_batch_3.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_3.xlsx'))
ABT_BATCH_3 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_3.xlsx')
df_abt_batch_4 = df_full_abt[3000:4000]
df_abt_batch_4.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_4.xlsx'))
ABT_BATCH_4 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_4.xlsx')
df_abt_batch_5 = df_full_abt[4000:5000]
df_abt_batch_5.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_5.xlsx'))
ABT_BATCH_5 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_5.xlsx')
df_abt_batch_6 = df_full_abt[5000:6000]
df_abt_batch_6.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_6.xlsx'))
ABT_BATCH_6 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_6.xlsx')
df_abt_batch_7 = df_full_abt[6000:7000]
df_abt_batch_7.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_7.xlsx'))
ABT_BATCH_7 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_7.xlsx')
df_abt_batch_8 = df_full_abt[7000:8000]
df_abt_batch_8.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_8.xlsx'))
ABT_BATCH_8 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_8.xlsx')
df_abt_batch_9 = df_full_abt[8000:9000]
df_abt_batch_9.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_9.xlsx'))
ABT_BATCH_9 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_9.xlsx')
df_abt_batch_10 = df_full_abt[9000:9742]
df_abt_batch_10.to_excel(os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_10.xlsx'))
ABT_BATCH_10 = os.path.join(DATA_FOLDER, 'movie_abt_enriched_FINAL_batch_10.xlsx')
df_abt_batch_10

,movieId,title,movielens_genres,movielens_rating,movielens_viewers,tmdb_Id,adult,backdrop_path,belongs_to_collection,budget,...,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,TMDB vote_average,TMDB vote_count
9000,139857,Colonia,Thriller,1.50,1.0,318781.0,0.0,/6gtlv8NYhzEeRyfbIbfeXp5KVJR.jpg,NaN,14000000.0,...,"[{'id': 2094, 'logo_path': None, 'name': 'Maje...","[{'iso_3166_1': 'DE', 'name': 'Germany'}, {'is...",2016-02-12,3621046.0,106.0,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,There is no turning back,7.259,1963.0
9001,139859,Ghost in the Shell: Arise - Border 2: Ghost Wh...,Action|Animation|Sci-Fi|Thriller,2.50,1.0,212168.0,0.0,/3LVz65LojUy53qGSbuUWZWCJXXD.jpg,"{'id': 196753, 'name': 'Ghost in the Shell: Ar...",0.0,...,"[{'id': 528, 'logo_path': '/fO3Aof3lXQclYpBByY...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",2013-11-29,0.0,57.0,"[{'english_name': 'Japanese', 'iso_639_1': 'ja...",Released,NaN,7.000,158.0
9002,139915,Some Kind of Beautiful,Comedy|Romance,2.25,2.0,253344.0,0.0,/fisrVuzjLMFsQP9VZbjMlwGF4Yr.jpg,NaN,0.0,...,"[{'id': 30131, 'logo_path': None, 'name': 'Sou...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",2015-07-16,2400000.0,99.0,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Every good love story has three sides.,5.645,601.0
9003,139994,Who's Afraid of Dracula?,Comedy|Horror,4.00,1.0,30459.0,0.0,/akbJi9TNHuZ5XAue4ZH0S6Oy5S3.jpg,"{'id': 704004, 'name': 'Fracchia Collection', ...",0.0,...,"[{'id': 5051, 'logo_path': None, 'name': 'Faso...","[{'iso_3166_1': 'IT', 'name': 'Italy'}]",1985-12-20,0.0,88.0,"[{'english_name': 'Italian', 'iso_639_1': 'it'...",Released,NaN,6.100,219.0
9004,140016,Always Watching: A Marble Hornets Story,Horror,2.00,1.0,329289.0,0.0,/m5hfYN8M4ZAWdpOznk1cqBjfvUz.jpg,NaN,0.0,...,"[{'id': 17393, 'logo_path': '/k28sCof6kheeOicL...","[{'iso_3166_1': 'US', 'name': 'United States o...",2015-04-07,714058.0,91.0,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,You shouldn't have looked.,5.162,170.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,Action|Animation|Comedy|Fantasy,4.00,1.0,432131.0,0.0,/clFQR5zwzDrayMVVe7axo0YLFfh.jpg,"{'id': 511109, 'name': 'Black Butler Collectio...",0.0,...,"[{'id': 13113, 'logo_path': '/xV5tPYKZhP2Ko9dO...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",2017-01-21,511132.0,100.0,"[{'english_name': 'Japanese', 'iso_639_1': 'ja...",Released,NaN,7.500,98.0
9738,193583,"No Game, No Life: Zero",Animation|Comedy|Fantasy,3.50,1.0,445030.0,0.0,/b0dP4lPgK8Dg0tQEPx6z73jRURA.jpg,NaN,0.0,...,"[{'id': 3464, 'logo_path': '/9k0nr75nwnNeT2MHe...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",2017-07-15,6356284.0,106.0,"[{'english_name': 'Japanese', 'iso_639_1': 'ja...",Released,NaN,7.800,426.0
9739,193585,Flint,Drama,3.50,1.0,479308.0,0.0,/tcM6XoTYKATJyNTIuGPk4yh0g6N.jpg,NaN,0.0,...,"[{'id': 11073, 'logo_path': '/wHs44fktdoj6c378...","[{'iso_3166_1': 'US', 'name': 'United States o...",2017-10-28,0.0,96.0,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,NaN,6.800,18.0
9740,193587,Bungo Stray Dogs: Dead Apple,Action|Animation,3.50,1.0,483455.0,0.0,/vjnS4iu0SdwXm2LLqZGNCfpId9t.jpg,"{'id': 1616714, 'name': 'Bungo Stray Dogs Coll...",0.0,...,"[{'id': 2849, 'logo_path': '/cN3JNIKawjOMtfk0G...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",2018-03-03,2373203.0,90.0,"[{'english_name': 'Japanese', 'iso_639_1': 'ja...",Released,An underworld is rising.,8.125,184.0


### Load Taxonomy as CSV and convert to JSON

In [5]:
df_taxonomy = pd.read_excel(TAXONOMY_FILE, sheet_name="custom_genre")
taxonomy_json = df_taxonomy.to_dict(orient='records')
taxonomy_json

[{'cluster_id': 5,
  'custom_genre': 'Clever Heist Capers',
  'description': 'This genre unites films centered around smart, stylish criminals and their intricate schemes, blending suspense, wit, and high-stakes cons or heists. These movies focus on the thrill of the plan, the dynamics within the crew, and the clever twists that keep audiences guessing.',
  'exemplar_overview': 'When a seasoned thief assembles a diverse team to pull off the ultimate heist, they must outsmart rival criminals and law enforcement alike. As tensions rise and loyalties are tested, the crew navigates double-crosses and unexpected obstacles to claim their prize and secure their freedom.',
  'included_movies': '[\'Heist\', \'To Catch a Thief\', \'Trading Places\', \'Heat\', "White Men Can\'t Jump", \'Gone in Sixty Seconds\', \'The Italian Job\', \'The Town\', \'Small Time Crooks\', "Ocean\'s Eleven", \'Inside Man\', \'Logan Lucky\']'},
 {'cluster_id': 8,
  'custom_genre': 'Global Espionage Thrillers',
  'descr

## Load ABT with row level chunking (Sandbox Method)

We are experimenting with chunking without embedding.  Which means it will use excel rows to find movies.

In [15]:
embedder = AzureOpenAIEmbedder(
    id="text-embedding-3-small",
    enable_batch=True,
    batch_size=10,
)

vector_db = ChromaDb(
    collection="excel_data",
    path="tmp/chromadb",
    persistent_client=False,
    embedder=embedder
)

knowledge = Knowledge(vector_db=vector_db)

# Use skip_if_exists=True so re-running resumes from where it left off
# If you have run the notebook and have a ChromaDB file already on your disk, skip knowledge.insert
knowledge.insert(
    path= ABT_BATCH_1,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)



INFO Adding content from path, 3e8b2b18-aeae-55c3-8889-4cbe092762a4, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_1.xlsx, None

INFO Upserting 1001 documents

In [17]:
# Use skip_if_exists=True so re-running resumes from where it left off
# If you have run the notebook and have a ChromaDB file already on your disk, skip knowledge.insert
knowledge.insert(
    path= ABT_BATCH_2,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

INFO Adding content from path, ccb61b90-166b-5c4d-906a-355ace155615, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_2.xlsx, None

INFO Upserting 1001 documents

In [18]:
agent = Agent(
    knowledge=knowledge,
    search_knowledge=True,
)

response = agent.run("""Provide all details on all Toy Story movies using the knowledge provided to you.
                     Your output should be a JSON covering all the keys provided to you.  
                     Do not skip any keys.  
                     Do not use outside information""", markdown=True)
print(response.content)

INFO Setting default model to OpenAI Chat

INFO Found 10 documents

INFO Found 10 documents

INFO Found 10 documents

INFO Found 10 documents

Here are the details of all Toy Story movies based on the knowledge available:

```json
[
    {
        "id": 1,
        "title": "Toy Story",
        "genres": "Adventure|Animation|Children|Comedy|Fantasy",
        "average_rating": 3.92,
        "total_votes": 215,
        "film_id": 862,
        "runtime": 81,
        "poster_path": "/bSoOThXLrbgtphS2eZjwy2xYwVt.jpg",
        "movie_collection": {
            "id": 10194,
            "name": "Toy Story Collection",
            "poster_path": "/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg",
            "backdrop_path": "/hApclyB9NEZEQujAVajzi5iWE4a.jpg"
        },
        "budget": 30000000,
        "genres_list": [
            {"id": 10751, "name": "Family"},
            {"id": 35, "name": "Comedy"},
            {"id": 16, "name": "Animation"},
            {"id": 12, "name": "Adventure"}
        ],
        "website": "http://toystory.disney.com/toy-story",
        "imdb_id": "tt0114709",
        "original_language": "en",
        "title_original

In [19]:
agent = Agent(
    knowledge=knowledge,
    search_knowledge=True,
)

response = agent.run("""Provide all details on all Godzilla movies using the knowledge provided to you.
                     Your output should be a JSON covering all the keys provided to you.  
                     Do not skip any keys.  
                     Do not use outside information""", markdown=True)
print(response.content)

INFO Setting default model to OpenAI Chat

INFO Found 10 documents

Here's a JSON formatted summary of some Godzilla movies, containing available details without any omissions:

```json
[
    {
        "id": 2363,
        "title": "Godzilla",
        "genres": ["Drama", "Horror", "Sci-Fi"],
        "popularity": 2.17,
        "vote_count": 3,
        "vote_average": 1678,
        "adult": false,
        "poster_path": "/UD0PBqGgXJOOqsNMOHeZ9Apjde.jpg",
        "collection": {
            "id": 374509,
            "name": "Godzilla (Showa) Collection",
            "poster_path": "/scvwS6k8gIW8w24UcmePQqVL10l.jpg",
            "backdrop_path": "/a20d9jnAidiWXoN4EjYdbW54OZ3.jpg"
        },
        "budget": 60000,
        "genres_list": [
            {"id": 53, "name": "Thriller"},
            {"id": 27, "name": "Horror"},
            {"id": 878, "name": "Science Fiction"}
        ],
        "imdb_id": "tt0047034",
        "original_language": "ja",
        "original_title": "ゴジラ",
        "overview": "Japan is thrown into a panic after several ships are 

In [20]:
 # Use skip_if_exists=True so re-running resumes from where it left off
# If you have run the notebook and have a ChromaDB file already on your disk, skip knowledge.insert
knowledge.insert(
    path= ABT_BATCH_3,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

knowledge.insert(
    path= ABT_BATCH_4,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

INFO Adding content from path, d8347c1e-adff-544d-9b40-f04200e76e39, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_3.xlsx, None

INFO Upserting 1001 documents

INFO Adding content from path, e142d515-f8a5-58d4-85ba-04284c2740d1, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_4.xlsx, None

INFO Upserting 1001 documents

In [21]:
# Use skip_if_exists=True so re-running resumes from where it left off
# If you have run the notebook and have a ChromaDB file already on your disk, skip knowledge.insert
knowledge.insert(
    path= ABT_BATCH_5,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

knowledge.insert(
    path= ABT_BATCH_6,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

INFO Adding content from path, a323ab27-6c67-5a40-b2ab-ca307aae6f7e, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_5.xlsx, None

INFO Upserting 1001 documents

INFO Adding content from path, 9f90f00a-8912-57ad-a07b-a62a93030bf6, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_6.xlsx, None

INFO Upserting 1001 documents

In [22]:
# Use skip_if_exists=True so re-running resumes from where it left off
# If you have run the notebook and have a ChromaDB file already on your disk, skip knowledge.insert
knowledge.insert(
    path= ABT_BATCH_7,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

knowledge.insert(
    path= ABT_BATCH_8,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

INFO Adding content from path, 0d3e7293-adb0-5ca5-afa9-0aa20a4560f2, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_7.xlsx, None

INFO Upserting 1001 documents

INFO Adding content from path, 189ac33c-34af-517b-876b-36e9b273dcf5, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_8.xlsx, None

INFO Upserting 1001 documents

In [23]:
# Use skip_if_exists=True so re-running resumes from where it left off
# If you have run the notebook and have a ChromaDB file already on your disk, skip knowledge.insert
knowledge.insert(
    path= ABT_BATCH_9,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

knowledge.insert(
    path= ABT_BATCH_10,
    reader=ExcelReader(chunking_strategy=RowChunking()),
    skip_if_exists=True,
)

INFO Adding content from path, c8a2b522-844f-5d2d-8900-97a04bb852da, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_9.xlsx, None

INFO Upserting 1001 documents

INFO Adding content from path, 4c2db154-709f-5fa7-912d-6c03c7777dc8, None,                                         
     ../data/input/movie_abt_enriched_FINAL_batch_10.xlsx, None

INFO Upserting 743 documents

In [13]:
knowledge

Knowledge(name=None, description=None, vector_db=<agno.vectordb.chroma.chromadb.ChromaDb object at 0x13ba1bf90>, contents_db=None, max_results=10, readers={}, content_sources=None, isolate_vector_search=False)

In [12]:
df_instructions = pd.read_excel(USE_CASE_FILE, sheet_name='Agent Instructions')
agent_instructions = df_instructions[df_instructions['Prompt Type'] == 'agent_prompt'].values[0][1]
AGENT_SYSTEM_PROMPT =  agent_instructions + "\n\nCustom Genre\n" + json.dumps(taxonomy_json, indent=2)
print('System prompt loaded:')
print(AGENT_SYSTEM_PROMPT)




System prompt loaded:
You are a friendly and knowledgeable movie expert. Your ONLY role is to help users discover and learn about movies. Do not answer questions unrelated to movies.

UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief, engaging conversation. Ask ONE leading question at a time across these dimensions (do not ask all at once):
  • Viewing context: Are you watching alone, with family, friends, or a date?
  • Audience: Who is the audience — adults, teens, young children, mixed group?
  • Genre mood: What genre or emotional mood are you in? (e.g., action, comedy, thriller, romance, drama, sci-fi, documentary)
  • Decade/era: Any preference for era or decade? (classic, 1980s–90s, modern, recent)
  • Country/language: Any preference for country of origin or spoken language?
  • Oscar/award interest: Are you interested in critically acclaimed or award-nominated films?
Stop asking when you have enough to make a pers

In [13]:
agent = Agent(
    model=model,
    db=agent_db,
    knowledge=knowledge,
    tools=[SerperTools()],
    session_id="My serper chatbot March 2",
    add_history_to_context=True,
    instructions=AGENT_SYSTEM_PROMPT + "\nCurrent memory is: {chat_history}",
    markdown=True,
    debug_mode=False
)




In [14]:
# Normally we use chat bot like a query engine.  Here is a way to test some queries.
test_queries = [
    "Recommend a romantic comedy for date night.",
    "Who are the nominees for Best Picture at the 2026 Oscars?",
    "What are the latest Bollywood movies in 2025?",
    "Is Oppenheimer available on Netflix?",
    "Summarize the chat so far"
    ]

for q in test_queries:
    print(f"\nQuery: {q}")
    response = agent.run(q, stream=False)
    print("\nBot:")
    print(response.content)
    print("-" * 80)


Query: Recommend a romantic comedy for date night.

Bot:
Great choice for a date night! To tailor a romantic comedy recommendation for you, could you share if you prefer a modern movie or maybe a classic romantic comedy? Or any other preference you have in mind for the tone or style?
--------------------------------------------------------------------------------

Query: Who are the nominees for Best Picture at the 2026 Oscars?

Bot:
It looks like you've asked again about the Best Picture nominees for the 2026 Oscars.

To confirm, some nominees include:
- Sinners
- One Battle After Another
- Frankenstein
- Einstein's Clocks
- The Stationmasters
- I Saw the TV Glow
- Rumi's Spin
- Canis Lupus
- Placid Edge

If you'd like more details about any of these films or want recommendations based on these nominees, just let me know!
--------------------------------------------------------------------------------

Query: What are the latest Bollywood movies in 2025?

Bot:
To recommend the latest

In [15]:
# This code will demonstrate a real flipped interaction

while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)


    # ✅ Retrieve latest stored memory
    chat_memory = agent.get_chat_history()


    print("\nSession Memory:")
    for m in chat_memory:
        print("-", m.content)


    print("\n" + "-"*50 + "\n")
#Initial query "Can you recommend a movie for a date night.  
# I love romantic movies and my spouse likes Samurai fiction"

Your input:  Can you recommend a movie for a date night.   # I love romantic movies and my spouse likes Samurai fiction

Agent:
Great! For your date night, you enjoy romantic movies, and your spouse is into samurai fiction. That's a wonderful combination.

Would you prefer a movie that leans more into romance with a touch of samurai elements, or a samurai film with some romantic storyline? Or maybe a perfect blend of both? This will help me find the ideal movie for both of you.

Session Memory:
- I would like to watch movie associated with current wars.  Which is the latest war 
- I can help you with that! To clarify, by "current wars," are you referring to ongoing or very recent conflicts that have been depicted in movies? Also, are you looking for movies based on a specific recent war, or are you open to films based on any contemporary conflicts? And lastly, do you have a preferred genre or style for these war-related movies (e.g., drama, documentary, action)?

Could you please share